# Módulo 11.2 — Graph Autoencoders
**Sergio Gevatschnaider** · red de seis actores ficticios, no relaciones reales.

Objetivo: entrenar un Graph Autoencoder sencillo, reconstruir enlaces y evitar fuga de aristas de prueba. El encoder usa convolución de grafo y el decoder producto interno.

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from torch import nn
torch.manual_seed(21);np.random.seed(21)
names=['IA A','IA B','IA C','Nube','Chips','Centro I+D']
features=torch.tensor([[1,.2],[.87,.35],[.75,.46],[.28,.9],[.37,.94],[.62,.7]],dtype=torch.float32)
positive=[(0,3),(0,4),(1,3),(1,4),(2,4),(2,5),(3,5)]
heldout=[(0,4),(1,4)]
train_pos=[e for e in positive if e not in heldout]
negatives=[(0,1),(0,2),(0,5),(1,2),(1,5),(2,3),(3,4),(4,5)]
train_neg=negatives[:6]; test_neg=negatives[6:]
def make_A(edges):
    A=torch.zeros((6,6))
    for i,j in edges: A[i,j]=A[j,i]=1
    return A
Atrain=make_A(train_pos)
I=torch.eye(6);Atilde=Atrain+I
D=torch.diag(torch.pow(Atilde.sum(1),-.5))
Anorm=D@Atilde@D
print('Aristas de entrenamiento',train_pos,'aristas de prueba',heldout)

## Entrenar el encoder con la adyacencia de entrenamiento, no la matriz completa
Las aristas de test no participan en la agregación. Se utiliza BCEWithLogitsLoss con pares observados de entrenamiento.

In [ ]:
class SmallGAE(nn.Module):
    def __init__(self):
        super().__init__(); self.W1=nn.Linear(2,8,bias=False); self.W2=nn.Linear(8,2,bias=False);self.bias=nn.Parameter(torch.tensor(-1.))
    def embed(self):return self.W2(torch.relu(Anorm@self.W1(features)))
    def logits(self,pairs):
        z=self.embed();return torch.stack([(z[i]*z[j]).sum()+self.bias for i,j in pairs])
model=SmallGAE();optimizer=torch.optim.Adam(model.parameters(),lr=.025,weight_decay=.001)
pairs=train_pos+train_neg
y=torch.tensor([1.]*len(train_pos)+[0.]*len(train_neg))
loss_fn=nn.BCEWithLogitsLoss();history=[]
for epoch in range(350):
    model.train();optimizer.zero_grad();loss=loss_fn(model.logits(pairs),y);loss.backward();optimizer.step();history.append(loss.item())
plt.plot(history);plt.xlabel('Época');plt.ylabel('BCE train');plt.show()

## Evaluación retenida y lectura crítica
El conjunto es minúsculo: los resultados no justifican conclusiones estadísticas ni inferencias sobre actores reales.

In [ ]:
model.eval()
with torch.no_grad():
    test_pairs=heldout+test_neg
    probs=torch.sigmoid(model.logits(test_pairs)).numpy()
print([(names[i],names[j],round(float(p),3)) for (i,j),p in zip(test_pairs,probs)])
from sklearn.metrics import roc_auc_score,average_precision_score
truth=np.array([1]*len(heldout)+[0]*len(test_neg))
print('AUC (muy inestable, n pequeño):',round(roc_auc_score(truth,probs),3))
print('AP (prevalencia artificial):',round(average_precision_score(truth,probs),3))

## Ejemplo conceptual de dos capas de relaciones
Cooperación y competencia deben codificarse mediante matrices separadas y requieren decoders por relación; el modelo anterior estima una sola capa.

In [ ]:
competition=[(0,1),(0,2),(1,2),(3,4)]
Acoord=make_A(positive);Acomp=make_A(competition)
print('Coordinación:',int(Acoord.sum().item()/2),'Competencia:',int(Acomp.sum().item()/2))
print('Son capas distintas:',not torch.equal(Acoord,Acomp))

## Preguntas de investigación
1. ¿Cómo definirías empíricamente cada relación con fuentes fechadas?
2. ¿Cómo controlarías la fuga de enlaces de test al encoder?
3. ¿Cuándo elegirías un decoder bilineal asimétrico?
4. ¿Qué ecuaciones dinámicas y observaciones necesitarías para contrastar Reina Roja?